# MTUS1 tertile analysis in TNBC cohorts

## Overview

This notebook compares tumours in the lowest and highest *MTUS1* expression tertiles across TNBC cohorts. It performs differential expression, Hallmark gene-set enrichment analysis, and figure-level summaries used to define recurrent pathway-level features of *MTUS1*-low TNBC.


In [ ]:
import scanpy as sc
import decoupler as dc
from anndata import AnnData
import random

# Import DESeq2
from pydeseq2.dds import DeseqDataSet, DefaultInference
from pydeseq2.ds import DeseqStats
from pydeseq2.preprocessing import deseq2_norm,deseq2_norm_fit,deseq2_norm_transform

# Only needed for processing
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans

from sklearn.metrics import silhouette_score

from statsmodels.stats.multitest import multipletests
from scipy.stats import ttest_ind

# batch correction
from inmoose.pycombat import pycombat_seq

from conorm import tmm,mrn,getmm

#from train import *
import sys
sys.path.append("..")  # Adds the parent directory to the system path
from src import utils
from src.utils import RNAseq_DataSet,pool_dataset

%load_ext autoreload
%autoreload 2


In [ ]:
# Analysis settings
name_gene = "MTUS1"
condition_a_tester = ["condition", "MTUS1-", "MTUS1+"]
condition_cancer = "TN"
name_TCGA = "TCGA-BRCA"
make_2_groups = "tertiles"


# 1. Load Data

In [ ]:
def load_bc_all():

    VUMC = RNAseq_DataSet(['VUMC'])
    # TNBC only
    VUMC.load_data()
    VUMC.group = VUMC.metadata

    SRP157974 = RNAseq_DataSet(['SRP157974'])
    # TNBC only
    SRP157974.load_data()
    SRP157974.group = SRP157974.metadata

    GSE181466 = RNAseq_DataSet(['GSE181466'])
    # TNBC only
    GSE181466.load_data()
    GSE181466.group = GSE181466.metadata

    SRP042620 = RNAseq_DataSet(['SRP042620'])
    SRP042620.load_data()
    SRP042620 = SRP042620.get_smaller_subset({'genes': None, 'patients': {'SUBTYPE': ['TRIPLE_NEGATIVE','ER_ENRICHED']}})
    SRP042620.group = SRP042620.metadata

    GSE192341 = RNAseq_DataSet(['GSE192341'])
    GSE192341.load_data()
    #GSE192341 = GSE192341.get_smaller_subset({'genes': None, 'patients': {'ER_FROM_IHC': ['ER_LOW'], 'PCR': ['PCR', 'noPCR']}})
    GSE192341.group = GSE192341.metadata

    #GSE123845 = RNAseq_DataSet(['GSE123845'])
    #GSE123845.load_data()
    #GSE123845 = GSE123845.get_smaller_subset({'genes': None, 'patients': {'TIME': ['DIAGNOSIS']}})
    #GSE123845.group = GSE123845.metadata

    GSE202203 = RNAseq_DataSet(['GSE202203'])
    GSE202203.load_data()
    GSE202203.features = GSE202203.features.loc[GSE202203.metadata.index]

    TCGA = RNAseq_DataSet([name_TCGA])
    TCGA.load_data()
    TCGA = TCGA.get_smaller_subset({'genes': None, 'patients': {'sample_type': ['Primary Tumor']}})
    TCGA.group = TCGA.metadata

    #VUMC_SRP042620_GSE181466_SRP157974_GSE192341_GSE202203_TCGA = pool_dataset({ 'VUMC':VUMC,'SRP042620':SRP042620,'SRP157974': SRP157974, 'GSE181466':GSE181466, 'GSE192341':GSE192341, 'GSE202203': GSE202203, 'TCGA': TCGA},'VUMC_SRP042620_GSE181466_SRP157974_GSE192341_GSE202203_TCGA')
    #VUMC_SRP042620_GSE181466_SRP157974_GSE192341_GSE202203_TCGA.group = VUMC_SRP042620_GSE181466_SRP157974_GSE192341_GSE202203_TCGA.metadata
    datasets = {'SRP042620':SRP042620,'GSE192341':GSE192341, 'TCGA': TCGA, 'GSE202203': GSE202203}#, 'VUMC_SRP042620_GSE181466_SRP157974_GSE192341_GSE202203_TCGA': VUMC_SRP042620_GSE181466_SRP157974_GSE192341_GSE202203_TCGA}
    datasets = {'SRP042620': SRP042620,  'VUMC':VUMC, 'GSE192341':GSE192341, 'GSE181466':GSE181466,'TCGA': TCGA, 'GSE202203': GSE202203, 'SRP157974':SRP157974}
    #VUMC_GSE192341_GSE123845_GSE202203_TCGA = pool_dataset({ 'VUMC':VUMC, 'GSE123845': GSE123845, 'GSE192341':GSE192341, 'GSE202203': GSE202203, 'TCGA': TCGA},'VUMC_GSE192341_GSE123845_GSE202203_TCGA')
    #VUMC_GSE192341_GSE123845_GSE202203_TCGA.group = VUMC_GSE192341_GSE123845_GSE202203_TCGA.metadata
    #datasets = {'VUMC':VUMC, 'GSE192341':GSE192341,'GSE123845': GSE123845,  'GSE202203': GSE202203, 'TCGA': TCGA,'VUMC_GSE192341_GSE123845_GSE202203_TCGA': VUMC_GSE192341_GSE123845_GSE202203_TCGA}
    return datasets

def load_tnbc_only():
        VUMC = RNAseq_DataSet(['VUMC'])
        # TNBC only
        VUMC.load_data()
        VUMC.group = VUMC.metadata

        SRP157974 = RNAseq_DataSet(['SRP157974'])
        # TNBC only
        SRP157974.load_data()
        SRP157974.group = SRP157974.metadata

        GSE181466 = RNAseq_DataSet(['GSE181466'])
        # TNBC only
        GSE181466.load_data()
        GSE181466.group = GSE181466.metadata

        # NO because MTUS1 is low for all samples in this dataset
        #GSE163882 = RNAseq_DataSet(['GSE163882'])
        #GSE163882.load_data()
        #GSE163882 = GSE163882.get_smaller_subset({'genes': None, 'patients': {'ER_FROM_IHC': ['ER_LOW'], 'HER2_FROM_IHC': ['HER2_LOW'], 'PR_FROM_IHC': ['PR_LOW']}})
        #GSE163882.group = GSE163882.metadata

        # No beccause PDX models only and no patient data
        #GSE235167 = RNAseq_DataSet(['GSE235167'])
        #GSE235167.load_data()
        #GSE235167 = GSE235167.get_smaller_subset({'genes': None, 'patients': SUBTYPE': ['TRIPLE_NEGATIVE']}})
        #GSE235167.group = GSE235167.metadata

        SRP042620 = RNAseq_DataSet(['SRP042620'])
        SRP042620.load_data()
        SRP042620 = SRP042620.get_smaller_subset({'genes': None, 'patients': {'SUBTYPE': ['TRIPLE_NEGATIVE',]}})
        SRP042620.group = SRP042620.metadata

        # seulement 6 samples TNBC
        #SRP032789 = RNAseq_DataSet(['SRP032789'])
        #SRP032789.load_data()
        #SRP032789 = SRP032789.get_smaller_subset({'genes': None, 'patients': {'SUBTYPE': ['TRIPLE_NEGATIVE']}})
        #SRP032789.group = SRP032789.metadata

        # Do not keep RSEM values.
        #GSE123845 = RNAseq_DataSet(['GSE123845'])
        #GSE123845.load_data()
        #GSE123845 = GSE123845.get_smaller_subset({'genes': None, 'patients': {'SUBTYPE': ['TRIPLE_NEGATIVE'],'TIME': ['DIAGNOSIS']}})
        #GSE123845.group = GSE123845.metadata

        GSE192341 = RNAseq_DataSet(['GSE192341'])
        GSE192341.load_data()
        GSE192341 = GSE192341.get_smaller_subset({'genes': None, 'patients': {'ER_FROM_IHC': ['ER_LOW']}})
        GSE192341.group = GSE192341.metadata
        
    
        GSE202203 = RNAseq_DataSet(['GSE202203'])
        GSE202203.load_data()
        GSE202203.features = GSE202203.features.loc[GSE202203.metadata.index]
        GSE202203 = GSE202203.get_smaller_subset({'genes': None, 'patients': {'SUBTYPE': ['TRIPLE_NEGATIVE']}})
        GSE202203.group = GSE202203.metadata

        TCGA = RNAseq_DataSet(['TCGA-BRCA'])
        TCGA.load_data()
        TCGA = TCGA.get_smaller_subset({'genes': None, 'patients': {'ER_FROM_IHC': ['ER_LOW'], 'HER2_FROM_IHC': ['HER2_LOW'], 'PR_FROM_IHC': ['PR_LOW'], 'sample_type': ['Primary Tumor']}})
        TCGA.group = TCGA.metadata

        # without GSE123845
        #VUMC_GSE192341_GSE202203_TCGA = pool_dataset({ 'VUMC':VUMC,  'GSE192341':GSE192341, 'GSE202203': GSE202203, 'TCGA': TCGA},'VUMC_GSE192341_GSE123845_GSE202203_TCGA')
        #VUMC_GSE192341_GSE202203_TCGA.group = VUMC_GSE192341_GSE202203_TCGA.metadata
        #datasets = {'VUMC':VUMC, 'GSE192341':GSE192341,  'GSE202203': GSE202203, 'TCGA': TCGA,'VUMC_GSE192341_GSE202203_TCGA': VUMC_GSE192341_GSE202203_TCGA}
        
        # without GSE123845 and with SRP157974
        #VUMC_SRP157974_GSE192341_GSE202203_TCGA = pool_dataset({ 'VUMC':VUMC, 'SRP157974':SRP157974,  'GSE192341':GSE192341, 'GSE202203': GSE202203, 'TCGA': TCGA},'VUMC_SRP157974_GSE192341_GSE202203_TCGA')
        #VUMC_SRP157974_GSE192341_GSE202203_TCGA.group = VUMC_SRP157974_GSE192341_GSE202203_TCGA.metadata
        #datasets = {'VUMC':VUMC, 'SRP157974':SRP157974, 'GSE192341':GSE192341,  'GSE202203': GSE202203, 'TCGA': TCGA,'VUMC_SRP157974_GSE192341_GSE202203_TCGA': VUMC_SRP157974_GSE192341_GSE202203_TCGA }


        #VUMC_GSE181466_SRP157974_GSE192341_SRP042620_GSE202203_TCGA = pool_dataset({'GSE181466':GSE181466, 'SRP157974':SRP157974, 'VUMC':VUMC, 'SRP042620': SRP042620, 'GSE192341':GSE192341, 'GSE202203': GSE202203, 'TCGA': TCGA},'VUMC_GSE181466_SRP157974_GSE192341_SRP042620_GSE202203_TCGA')
        #VUMC_GSE181466_SRP157974_GSE192341_SRP042620_GSE202203_TCGA.group = VUMC_GSE181466_SRP157974_GSE192341_SRP042620_GSE202203_TCGA.metadata
        datasets = {'SRP042620': SRP042620,  'VUMC':VUMC, 'GSE192341':GSE192341, 'GSE181466':GSE181466,'TCGA': TCGA, 'GSE202203': GSE202203, 'SRP157974':SRP157974}#, 'VUMC_GSE181466_SRP157974_GSE192341_SRP042620_GSE202203_TCGA': VUMC_GSE181466_SRP157974_GSE192341_SRP042620_GSE202203_TCGA}
        return datasets

def load_all_cancer():
    
    TCGA_LIHC = RNAseq_DataSet(['TCGA-LIHC']) #?
    TCGA_LIHC.load_data()
    TCGA_LIHC = TCGA_LIHC.get_smaller_subset({'genes': None, 'patients': {'sample_type': ['Primary Tumor']}})

    TCGA_LUAD = RNAseq_DataSet(['TCGA-LUAD']) #?
    TCGA_LUAD.load_data()
    TCGA_LUAD = TCGA_LUAD.get_smaller_subset({'genes': None, 'patients': {'sample_type': ['Primary Tumor']}})

    TCGA_LUSC = RNAseq_DataSet(['TCGA-LUSC']) #oui
    TCGA_LUSC.load_data()
    TCGA_LUSC = TCGA_LUSC.get_smaller_subset({'genes': None, 'patients': {'sample_type': ['Primary Tumor']}})

    TCGA_LGG = RNAseq_DataSet(['TCGA-LGG']) #?
    TCGA_LGG.load_data()
    TCGA_LGG = TCGA_LGG.get_smaller_subset({'genes': None, 'patients': {'sample_type': ['Primary Tumor']}})

    TCGA_COAD = RNAseq_DataSet(['TCGA-COAD']) #ok
    TCGA_COAD.load_data()
    TCGA_COAD = TCGA_COAD.get_smaller_subset({'genes': None, 'patients': {'sample_type': ['Primary Tumor']}})

    TCGA_PAAD = RNAseq_DataSet(['TCGA-PAAD']) #?
    TCGA_PAAD.load_data()
    TCGA_PAAD = TCGA_PAAD.get_smaller_subset({'genes': None, 'patients': {'sample_type': ['Primary Tumor']}})

    TCGA_OV = RNAseq_DataSet(['TCGA-OV']) #?
    TCGA_OV.load_data()
    TCGA_OV = TCGA_OV.get_smaller_subset({'genes': None, 'patients': {'sample_type': ['Primary Tumor']}})

    #TCGA_CHOL = RNAseq_DataSet(['TCGA-CHOL']) #NON
    #TCGA_CHOL.load_data()
    #TCGA_CHOL = TCGA_CHOL.get_smaller_subset({'genes': None, 'patients': {'sample_type': ['Primary Tumor']}})

    #####
    #TCGA_KICH = RNAseq_DataSet(['TCGA-KICH'])#NON
    #TCGA_KICH.load_data()
    #TCGA_KICH = TCGA_KICH.get_smaller_subset({'genes': None, 'patients': {'sample_type': ['Primary Tumor']}})

    TCGA_KIRP = RNAseq_DataSet(['TCGA-KIRP'])#OK
    TCGA_KIRP.load_data()
    TCGA_KIRP = TCGA_KIRP.get_smaller_subset({'genes': None, 'patients': {'sample_type': ['Primary Tumor']}})

    TCGA_KIRC = RNAseq_DataSet(['TCGA-KIRC'])#ok
    TCGA_KIRC.load_data()
    TCGA_KIRC = TCGA_KIRC.get_smaller_subset({'genes': None, 'patients': {'sample_type': ['Primary Tumor']}})

    TCGA_BLCA = RNAseq_DataSet(['TCGA-BLCA']) #ok
    TCGA_BLCA.load_data()
    TCGA_BLCA = TCGA_BLCA.get_smaller_subset({'genes': None, 'patients': {'sample_type': ['Primary Tumor']}})

    TCGA_HNSC = RNAseq_DataSet(['TCGA-HNSC'])#?
    TCGA_HNSC.load_data()
    TCGA_HNSC = TCGA_HNSC.get_smaller_subset({'genes': None, 'patients': {'sample_type': ['Primary Tumor']}})

    #TCGA_THCA = RNAseq_DataSet(['TCGA-THCA']) #NON
    #TCGA_THCA.load_data()
    #TCGA_THCA = TCGA_THCA.get_smaller_subset({'genes': None, 'patients': {'sample_type': ['Primary Tumor']}})

    TCGA_PRAD = RNAseq_DataSet(['TCGA-PRAD']) #?
    TCGA_PRAD.load_data()
    TCGA_PRAD = TCGA_PRAD.get_smaller_subset({'genes': None, 'patients': {'sample_type': ['Primary Tumor']}})

    TCGA_READ = RNAseq_DataSet(['TCGA-READ']) #OK
    TCGA_READ.load_data()
    TCGA_READ = TCGA_READ.get_smaller_subset({'genes': None, 'patients': {'sample_type': ['Primary Tumor']}})

    TCGA_UCEC = RNAseq_DataSet(['TCGA-UCEC']) #OK
    TCGA_UCEC.load_data()
    TCGA_UCEC = TCGA_UCEC.get_smaller_subset({'genes': None, 'patients': {'sample_type': ['Primary Tumor']}})

    #TCGA_STAD = RNAseq_DataSet(['TCGA-STAD']) #NON
    #TCGA_STAD.load_data()
    #TCGA_STAD = TCGA_STAD.get_smaller_subset({'genes': None, 'patients': {'sample_type': ['Primary Tumor']}})

    # BC
    VUMCa = RNAseq_DataSet(['VUMC'])
    # TNBC only
    VUMCa.load_data()
    VUMCa.group = VUMCa.metadata

    SRP157974a = RNAseq_DataSet(['SRP157974'])
    # TNBC only
    SRP157974a.load_data()
    SRP157974a.group = SRP157974a.metadata

    GSE181466a = RNAseq_DataSet(['GSE181466'])
    # TNBC only
    GSE181466a.load_data()
    GSE181466a.group = GSE181466a.metadata

    SRP042620a = RNAseq_DataSet(['SRP042620'])
    SRP042620a.load_data()
    SRP042620a = SRP042620a.get_smaller_subset({'genes': None, 'patients': {'SUBTYPE': ['TRIPLE_NEGATIVE','ER_ENRICHED']}})
    SRP042620a.group = SRP042620a.metadata

    GSE192341a = RNAseq_DataSet(['GSE192341'])
    GSE192341a.load_data()
    GSE192341a.group = GSE192341a.metadata

    #GSE123845 = RNAseq_DataSet(['GSE123845'])
    #GSE123845.load_data()
    #GSE123845 = GSE123845.get_smaller_subset({'genes': None, 'patients': {'TIME': ['DIAGNOSIS']}})
    #GSE123845.group = GSE123845.metadata

    GSE202203a = RNAseq_DataSet(['GSE202203'])
    GSE202203a.load_data()
    GSE202203a.features = GSE202203a.features.loc[GSE202203a.metadata.index]

    TCGAa = RNAseq_DataSet([name_TCGA])
    TCGAa.load_data()
    TCGAa = TCGAa.get_smaller_subset({'genes': None, 'patients': {'sample_type': ['Primary Tumor']}})
    TCGAa.group = TCGAa.metadata

    # TN
    VUMC = RNAseq_DataSet(['VUMC'])
    # TNBC only
    VUMC.load_data()
    VUMC.group = VUMC.metadata

    SRP157974 = RNAseq_DataSet(['SRP157974'])
    # TNBC only
    SRP157974.load_data()
    SRP157974.group = SRP157974.metadata

    GSE181466 = RNAseq_DataSet(['GSE181466'])
    # TNBC only
    GSE181466.load_data()
    GSE181466.group = GSE181466.metadata

    SRP042620 = RNAseq_DataSet(['SRP042620'])
    SRP042620.load_data()
    SRP042620 = SRP042620.get_smaller_subset({'genes': None, 'patients': {'SUBTYPE': ['TRIPLE_NEGATIVE',]}})
    SRP042620.group = SRP042620.metadata

    GSE192341 = RNAseq_DataSet(['GSE192341'])
    GSE192341.load_data()
    GSE192341 = GSE192341.get_smaller_subset({'genes': None, 'patients': {'ER_FROM_IHC': ['ER_LOW']}})
    GSE192341.group = GSE192341.metadata
    

    GSE202203 = RNAseq_DataSet(['GSE202203'])
    GSE202203.load_data()
    GSE202203.features = GSE202203.features.loc[GSE202203.metadata.index]
    GSE202203 = GSE202203.get_smaller_subset({'genes': None, 'patients': {'SUBTYPE': ['TRIPLE_NEGATIVE']}})
    GSE202203.group = GSE202203.metadata

    TCGA = RNAseq_DataSet(['TCGA-BRCA'])
    TCGA.load_data()
    TCGA = TCGA.get_smaller_subset({'genes': None, 'patients': {'ER_FROM_IHC': ['ER_LOW'], 'HER2_FROM_IHC': ['HER2_LOW'], 'PR_FROM_IHC': ['PR_LOW'], 'sample_type': ['Primary Tumor']}})
    TCGA.group = TCGA.metadata

    #datasets = {'TCGA-LUAD': TCGA_LUAD, 'TCGA-LUSC': TCGA_LUSC,'TCGA-KIRC': TCGA_KIRC,'TCGA-COAD': TCGA_COAD, 'TCGA-PAAD': TCGA_PAAD, 'TCGA-OV': TCGA_OV,  'GSE123845_GSE202203_TCGA': BC_all_pooled, 'VUMC_GSE192341_GSE123845_GSE202203_TCGA': TNBC_all_pooled }
    datasets = {'TCGA-KIRC': TCGA_KIRC,'TCGA-KIRP':TCGA_KIRP,'TCGA-BLCA':TCGA_BLCA,'TCGA-COAD': TCGA_COAD,'TCGA-READ':TCGA_READ, 'TCGA-PAAD': TCGA_PAAD, 'TCGA-LIHC':TCGA_LIHC,'TCGA-LGG':TCGA_LGG ,'TCGA-HNSC':TCGA_HNSC,'TCGA-LUAD': TCGA_LUAD, 'TCGA-LUSC': TCGA_LUSC,'TCGA-PRAD':TCGA_PRAD,  'TCGA-OV': TCGA_OV,'TCGA-UCEC': TCGA_UCEC,
                'VUMCa':VUMCa, 'SRP157974a':SRP157974a, 'GSE181466a':GSE181466a, 'SRP042620a': SRP042620a, 'GSE192341a':GSE192341a, 'GSE202203a': GSE202203a, 'TCGAa': TCGAa,    
                'VUMC':VUMC, 'SRP157974':SRP157974, 'GSE181466':GSE181466, 'SRP042620': SRP042620, 'GSE192341':GSE192341,  'GSE202203': GSE202203, 'TCGA': TCGA}#, 'VUMCa_GSE192341a_GSE123845a_GSE202203a_TCGAa': BC_all_pooled, 'VUMC_GSE192341_GSE123845_GSE202203_TCGA': TNBC_all_pooled } #'TCGA-CHOL':TCGA_CHOL,'TCGA-STAD':TCGA_STAD,'TCGA-THCA':TCGA_THCA,'TCGA-KICH':TCGA_KICH,
    return datasets

def load_bc_and_tn_separately():
    # BC
    
    VUMCa = RNAseq_DataSet(['VUMC'])
    # TNBC only
    VUMCa.load_data()
    VUMCa.group = VUMCa.metadata

    SRP157974a = RNAseq_DataSet(['SRP157974'])
    # TNBC only
    SRP157974a.load_data()
    SRP157974a.group = SRP157974a.metadata

    GSE181466a = RNAseq_DataSet(['GSE181466'])
    # TNBC only
    GSE181466a.load_data()
    GSE181466a.group = GSE181466a.metadata

    SRP042620a = RNAseq_DataSet(['SRP042620'])
    SRP042620a.load_data()
    SRP042620a = SRP042620a.get_smaller_subset({'genes': None, 'patients': {'SUBTYPE': ['TRIPLE_NEGATIVE','ER_ENRICHED']}})
    SRP042620a.group = SRP042620a.metadata

    GSE192341a = RNAseq_DataSet(['GSE192341'])
    GSE192341a.load_data()
    #GSE192341 = GSE192341.get_smaller_subset({'genes': None, 'patients': {'ER_FROM_IHC': ['ER_LOW'], 'PCR': ['PCR', 'noPCR']}})
    GSE192341a.group = GSE192341a.metadata

    #GSE123845 = RNAseq_DataSet(['GSE123845'])
    #GSE123845.load_data()
    #GSE123845 = GSE123845.get_smaller_subset({'genes': None, 'patients': {'TIME': ['DIAGNOSIS']}})
    #GSE123845.group = GSE123845.metadata

    GSE202203a = RNAseq_DataSet(['GSE202203'])
    GSE202203a.load_data()
    GSE202203a.features = GSE202203a.features.loc[GSE202203a.metadata.index]

    TCGAa = RNAseq_DataSet([name_TCGA])
    TCGAa.load_data()
    TCGAa = TCGAa.get_smaller_subset({'genes': None, 'patients': {'sample_type': ['Primary Tumor']}})
    TCGAa.group = TCGAa.metadata

    #BC_all_pooled = pool_dataset({ 'SRP042620a': SRP042620a,  'VUMCa':VUMCa, 'GSE192341a':GSE192341a, 'GSE181466a':GSE181466a,'TCGAa': TCGAa, 'GSE202203a': GSE202203a, 'SRP157974a':SRP157974a},'VUMCa_GSE192341a_GSE181466a_SRP042620a_GSE202203a_TCGAa_SRP157974a')

    # TN
    VUMC = RNAseq_DataSet(['VUMC'])
    # TNBC only
    VUMC.load_data()
    VUMC.group = VUMC.metadata

    SRP157974 = RNAseq_DataSet(['SRP157974'])
    # TNBC only
    SRP157974.load_data()
    SRP157974.group = SRP157974.metadata

    GSE181466 = RNAseq_DataSet(['GSE181466'])
    # TNBC only
    GSE181466.load_data()
    GSE181466.group = GSE181466.metadata

    SRP042620 = RNAseq_DataSet(['SRP042620'])
    SRP042620.load_data()
    SRP042620 = SRP042620.get_smaller_subset({'genes': None, 'patients': {'SUBTYPE': ['TRIPLE_NEGATIVE',]}})
    SRP042620.group = SRP042620.metadata

    GSE192341 = RNAseq_DataSet(['GSE192341'])
    GSE192341.load_data()
    GSE192341 = GSE192341.get_smaller_subset({'genes': None, 'patients': {'ER_FROM_IHC': ['ER_LOW']}})
    GSE192341.group = GSE192341.metadata
    

    GSE202203 = RNAseq_DataSet(['GSE202203'])
    GSE202203.load_data()
    GSE202203.features = GSE202203.features.loc[GSE202203.metadata.index]
    GSE202203 = GSE202203.get_smaller_subset({'genes': None, 'patients': {'SUBTYPE': ['TRIPLE_NEGATIVE']}})
    GSE202203.group = GSE202203.metadata

    TCGA = RNAseq_DataSet(['TCGA-BRCA'])
    TCGA.load_data()
    TCGA = TCGA.get_smaller_subset({'genes': None, 'patients': {'ER_FROM_IHC': ['ER_LOW'], 'HER2_FROM_IHC': ['HER2_LOW'], 'PR_FROM_IHC': ['PR_LOW'], 'sample_type': ['Primary Tumor']}})
    TCGA.group = TCGA.metadata

    #TNBC_all_pooled = pool_dataset({ 'VUMC':VUMC, 'GSE123845': GSE123845, 'GSE192341':GSE192341, 'GSE202203': GSE202203, 'TCGA': TCGA},'VUMC_GSE192341_GSE123845_GSE202203_TCGA')
    #TNBC_all_pooled.group = TNBC_all_pooled.metadata

    datasets = {'SRP042620a': SRP042620a,  'VUMCa':VUMCa, 'GSE192341a':GSE192341a, 'GSE181466a':GSE181466a,'TCGAa': TCGAa, 'GSE202203a': GSE202203a, 'SRP157974a':SRP157974a,'SRP042620': SRP042620,  'VUMC':VUMC, 'GSE192341':GSE192341, 'GSE181466':GSE181466,'TCGA': TCGA, 'GSE202203': GSE202203, 'SRP157974':SRP157974}#, 'BC_all_pooled': BC_all_pooled,'TNBC_all_pooled': TNBC_all_pooled}
    return datasets


if condition_cancer == "allBC":
    datasets = load_bc_all()
elif condition_cancer == "TN":
    datasets = load_tnbc_only()
elif condition_cancer == "all_cancer":
    datasets = load_all_cancer()
elif condition_cancer == "allBC+TN":
    datasets = load_bc_and_tn_separately()
else:
    raise ValueError(f"Unknown condition_cancer value: {condition_cancer}")

In [ ]:
for name,dst in datasets.items():
    print(name, ", number of samples: ", (dst.features.shape[0]))

# 2. Make 2 groups

## A. Tertile-based comparison groups

In [ ]:
if make_2_groups == 'median':
    for i, (name_dataset,dataset) in enumerate(datasets.items()):

        print(name_dataset)
        
        Raw_counts = dataset.features.round().astype(int)
        # Normalization
        normalize_counts_df = deseq2_norm(dataset.features)[0]
        vst_df = np.log2(normalize_counts_df + 1)

        t1 = vst_df[name_gene].median()
        dataset.metadata = dataset.metadata.copy()
        dataset.metadata['condition_transfer'] = [condition_a_tester[2] if x >= t1 else (condition_a_tester[1] if x<t1 else 'unknown') for x in vst_df[name_gene]]

        # Create anndata object
        dataset.anndata = AnnData(X=dataset.features.astype(int), obs=dataset.metadata)
        dataset.anndata.obs['condition'] = dataset.metadata['condition_transfer']
        adata = dataset.anndata

        # QUALITY CONTROL Obtain genes that pass the thresholds
        genes = dc.filter_by_expr(adata, group='condition', min_count=1, min_total_count=10, large_n=1, min_prop=1)
        # Filter by these genes
        adata = adata[:, genes].copy()
        dataset.anndata = adata
        print("*"*20)

In [ ]:
if make_2_groups == 'tertiles':
    for i, (name_dataset,dataset) in enumerate(datasets.items()):

        print(name_dataset)
        
        Raw_counts = dataset.features.round().astype(int)
        # Normalization
        normalize_counts_df = deseq2_norm(dataset.features)[0]
        vst_df = np.log2(normalize_counts_df + 1)

        t1 = vst_df[name_gene].quantile(1/3)
        t2 = vst_df[name_gene].quantile(2/3)
        dataset.metadata = dataset.metadata.copy()
        dataset.metadata['condition_transfer'] = [condition_a_tester[2] if x > t2 else (condition_a_tester[1] if x<t1 else 'unknown') for x in vst_df[name_gene]]

        # Create anndata object
        dataset.anndata = AnnData(X=dataset.features.astype(int), obs=dataset.metadata)
        dataset.anndata.obs['condition'] = dataset.metadata['condition_transfer']
        adata = dataset.anndata

        # QUALITY CONTROL Obtain genes that pass the thresholds
        genes = dc.filter_by_expr(adata, group='condition', min_count=1, min_total_count=10, large_n=1, min_prop=1)
        # Filter by these genes
        adata = adata[:, genes].copy()
        dataset.anndata = adata
        print("*"*20)


In [ ]:
if make_2_groups == 'quartiles_all':    
    for i, (name_dataset,dataset) in enumerate(datasets.items()):

        print(name_dataset)
        
        Raw_counts = dataset.features.round().astype(int)
        # Normalization
        normalize_counts_df = deseq2_norm(dataset.features)[0]
        vst_df = np.log2(normalize_counts_df + 1)

        t1 = vst_df[name_gene].quantile(1/4)
        t2 = vst_df[name_gene].quantile(3/4)
        dataset.metadata = dataset.metadata.copy()
        dataset.metadata['condition_transfer'] = [condition_a_tester[2] if x > t2 else (condition_a_tester[1] if x<t1 else 'unknown') for x in vst_df[name_gene]]

        # Create anndata object
        dataset.anndata = AnnData(X=dataset.features.astype(int), obs=dataset.metadata)
        dataset.anndata.obs['condition'] = dataset.metadata['condition_transfer']
        adata = dataset.anndata

        # QUALITY CONTROL Obtain genes that pass the thresholds
        genes = dc.filter_by_expr(adata, group='condition', min_count=1, min_total_count=10, large_n=1, min_prop=1)
        # Filter by these genes
        adata = adata[:, genes].copy()
        dataset.anndata = adata
        print("*"*20)


In [ ]:
# Use quartiles for large datasets and tertiles otherwise.
if make_2_groups == 'quartiles_big':    
    for i, (name_dataset,dataset) in enumerate(datasets.items()):

        print(name_dataset)
        
        Raw_counts = dataset.features.round().astype(int)
        # Normalization
        normalize_counts_df = deseq2_norm(dataset.features)[0]
        vst_df = np.log2(normalize_counts_df + 1)

        if dataset.features.shape[0] < 55:
            t1 = vst_df[name_gene].quantile(1/3)
            t2 = vst_df[name_gene].quantile(2/3)
        else:
            t1 = vst_df[name_gene].quantile(1/4)
            t2 = vst_df[name_gene].quantile(3/4)
        dataset.metadata = dataset.metadata.copy()
        dataset.metadata['condition_transfer'] = [condition_a_tester[2] if x > t2 else (condition_a_tester[1] if x<t1 else 'unknown') for x in vst_df[name_gene]]

        # Create anndata object
        dataset.anndata = AnnData(X=dataset.features.astype(int), obs=dataset.metadata)
        dataset.anndata.obs['condition'] = dataset.metadata['condition_transfer']
        adata = dataset.anndata

        # QUALITY CONTROL Obtain genes that pass the thresholds
        genes = dc.filter_by_expr(adata, group='condition', min_count=1, min_total_count=10, large_n=1, min_prop=1)
        # Filter by these genes
        adata = adata[:, genes].copy()
        dataset.anndata = adata
        print("*"*20)

# Pool datasets (after 2 groups)

In [ ]:
def pool_dataset_2(dict_dataset, pool_dataset_name):
    """
    Pool several datasets while retaining all metadata fields.
    - Concatenate count matrices by rows.
    - Concatenate metadata tables while automatically aligning columns.
    - Add a 'dataset' column to record the source dataset for each sample.
    """

    dst = RNAseq_DataSet([pool_dataset_name]) 
    Raw_counts = None
    metadata_ = pd.DataFrame()

    for dataset_name, dataset in dict_dataset.items():

        # --- 1) Counts ---
        counts = dataset.features.copy()
        counts = counts.round().astype(int)

        # Keep only shared genes.
        if Raw_counts is None:
            Raw_counts = counts
        else:
            common_genes = Raw_counts.columns.intersection(counts.columns)
            Raw_counts = Raw_counts[common_genes]
            counts = counts[common_genes]
            Raw_counts = pd.concat([Raw_counts, counts], axis=0)

        # --- 2) Metadata ---
        meta = dataset.metadata.copy()

        # Add source dataset information.
        meta["dataset"] = dataset_name

        # Concatenate while automatically aligning columns.
        metadata_ = pd.concat([metadata_, meta], axis=0, sort=False)

    # finalisation
    dst.features = Raw_counts
    dst.metadata = metadata_

    dst.common_predictors = list(dst.features.columns)
    dst.dataset_type = "RNA-seq:counts"

    # Create anndata object
    dst.metadata = dst.metadata.loc[dst.features.index].copy()
    dst.anndata = AnnData(X=dst.features.astype(int), obs=dst.metadata)
    dst.anndata.obs["condition_transfer"] = dst.anndata.obs["condition_transfer"].astype(str)
    dst.anndata.obs["condition"] = dst.anndata.obs["condition_transfer"]
    # QUALITY CONTROL Obtain genes that pass the thresholds
    genes = dc.filter_by_expr(dst.anndata, group='condition', min_count=1, min_total_count=10, large_n=1, min_prop=1)
    # Filter by these genes
    dst.anndata = dst.anndata[:, genes].copy()

    return dst

In [ ]:
if condition_cancer == "allBC":

    BC_All_Pooled = pool_dataset_2(datasets,'BC_All_Pooled')
    BC_All_Pooled.group = BC_All_Pooled.metadata
    datasets['BC_All_Pooled'] = BC_All_Pooled
elif condition_cancer == "TN":
    TNBC_All_Pooled = pool_dataset_2(datasets,'TNBC_All_Pooled')
    TNBC_All_Pooled.group = TNBC_All_Pooled.metadata
    datasets['TNBC_All_Pooled'] = TNBC_All_Pooled
        
elif condition_cancer == "all_cancer":
    #BC_pooled
    BC_All_Pooled = pool_dataset_2({'VUMCa': datasets['VUMCa'], 'SRP042620a': datasets['SRP042620a'], 'GSE181466a': datasets['GSE181466a'], 'SRP157974a': datasets['SRP157974a'], 'GSE192341a': datasets['GSE192341a'], 'GSE202203a': datasets['GSE202203a'], 'TCGAa': datasets['TCGAa']},'BC_All_Pooled')
    BC_All_Pooled.group = BC_All_Pooled.metadata
    datasets['BC_All_Pooled'] = BC_All_Pooled
    # delete previous pooled to save memory
    del datasets['VUMCa']
    del datasets['SRP042620a']
    del datasets['GSE181466a']
    del datasets['SRP157974a']
    del datasets['GSE192341a']
    del datasets['GSE202203a']
    del datasets['TCGAa']
    #TN_pooled
    TNBC_All_Pooled = pool_dataset_2({'VUMC': datasets['VUMC'], 'GSE181466': datasets['GSE181466'], 'SRP157974': datasets['SRP157974'], 'GSE192341': datasets['GSE192341'], 'SRP042620': datasets['SRP042620'], 'GSE202203': datasets['GSE202203'], 'TCGA': datasets['TCGA']},'TNBC_All_Pooled')
    TNBC_All_Pooled.group = TNBC_All_Pooled.metadata
    datasets['TNBC_All_Pooled'] = TNBC_All_Pooled
    # delete previous pooled to save memory
    del datasets['VUMC']
    del datasets['GSE181466']
    del datasets['SRP157974']
    del datasets['GSE192341']
    del datasets['SRP042620']
    del datasets['GSE202203']
    del datasets['TCGA']
    
    
elif condition_cancer == "all_cancer_and_BC":
    pass
elif condition_cancer == "allBC+TN":
    #BC_pooled
    BC_All_Pooled = pool_dataset_2({'VUMCa': datasets['VUMCa'], 'SRP042620a': datasets['SRP042620a'], 'GSE181466a': datasets['GSE181466a'], 'SRP157974a': datasets['SRP157974a'], 'GSE192341a': datasets['GSE192341a'], 'GSE202203a': datasets['GSE202203a'], 'TCGAa': datasets['TCGAa']},'BC_All_Pooled')
    BC_All_Pooled.group = BC_All_Pooled.metadata
    datasets['BC_All_Pooled'] = BC_All_Pooled
    #TN_pooled
    TNBC_All_Pooled = pool_dataset_2({'VUMC': datasets['VUMC'], 'GSE181466': datasets['GSE181466'], 'SRP157974': datasets['SRP157974'], 'GSE192341': datasets['GSE192341'], 'SRP042620': datasets['SRP042620'], 'GSE202203': datasets['GSE202203'], 'TCGA': datasets['TCGA']},'TNBC_All_Pooled')
    TNBC_All_Pooled.group = TNBC_All_Pooled.metadata
    datasets['TNBC_All_Pooled'] = TNBC_All_Pooled

else:
    raise ValueError(f"Unknown condition_cancer value: {condition_cancer}")

# 3. Differential expression

In [ ]:
import numpy as np
import pandas as pd
from scipy import sparse

from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
from pydeseq2.default_inference import DefaultInference


# =========================
# 1) Global batch variable
# =========================
def add_global_batch(dataset_name: str, dataset, tcga_min_n: int = 3):
    """
    Build adata.obs["batch_global"] with cohort-specific logic:

    - TCGA: use a collapsed within-cohort batch (batch_number_collapsed)
    - SRP157974: optionally use sra.library_selection (if present)
    - Others: use dataset name as a between-cohort batch
    """
    adata = dataset.anndata

    # Ensure COHORT and dataset labels exist
    if "COHORT" not in adata.obs.columns:
        adata.obs["COHORT"] = dataset_name
    if "dataset" not in adata.obs.columns:
        adata.obs["dataset"] = dataset_name

    # Default: batch is the dataset name
    adata.obs["batch_global"] = ("DS_" + adata.obs["dataset"].astype(str))

    # ---------- TCGA-specific: collapse rare batches ----------
    is_tcga = adata.obs["COHORT"].astype(str).str.contains("TCGA").any() or ("TCGA" in dataset_name)

    if is_tcga:
        # Ensure batch_number exists
        if "batch_number" not in adata.obs.columns:
            adata.obs["batch_number"] = "unknown"

        # Collapse rare batch levels (sizes computed across all samples currently in adata)
        # Note: requires "condition" to exist; if some samples have other condition labels,
        # they will still be included in crosstab counts.
        if "condition" in adata.obs.columns:
            tab = pd.crosstab(adata.obs["batch_number"], adata.obs["condition"])
            sizes = tab.sum(axis=1)
        else:
            # Fallback: just use raw batch_number if condition is missing
            sizes = adata.obs["batch_number"].value_counts()

        rare = sizes[sizes < tcga_min_n].index

        bn = adata.obs["batch_number"].astype(str)
        bn_collapsed = bn.where(~adata.obs["batch_number"].isin(rare),
                                other=f"TCGA_rare_lt{tcga_min_n}")

        adata.obs["batch_number_collapsed"] = pd.Categorical(bn_collapsed)

        # Use collapsed TCGA batch as the global batch
        adata.obs["batch_global"] = "TCGA_" + adata.obs["batch_number_collapsed"].astype(str)

    # ---------- SRP157974-specific: use library selection if available ----------
    if dataset_name == "SRP157974":
        if "sra.library_selection" in adata.obs.columns:
            adata.obs["batch_global"] = (
                "SRP157974_" + adata.obs["sra.library_selection"].astype(str)
            )
        else:
            # If the column is missing, keep the default dataset-level batch
            adata.obs["batch_global"] = "DS_" + adata.obs["dataset"].astype(str)

    # Cast to categorical for DESeq2
    adata.obs["batch_global"] = adata.obs["batch_global"].astype("category")

    return adata




# =========================
# 2) Clean obs for the design
# =========================
def clean_obs_for_design(adata):
    # Fail early if condition is missing
    if "condition" not in adata.obs.columns:
        raise KeyError("Missing 'condition' in adata.obs")

    obs = adata.obs.copy()

    # Keep only MTUS1+/MTUS1- samples
    obs = obs[obs["condition"].isin(["MTUS1+", "MTUS1-"])].copy()
    idx = obs.index

    # Subset X safely
    X = adata[idx, :].X
    if sparse.issparse(X):
        X = X.copy()
    else:
        X = np.asarray(X).copy()

    # Keep var
    var = adata.var.copy()

    # Rebuild a minimal AnnData (drops obsm/layers that may contain pandas.Series)
    adata2 = AnnData(X=X, obs=obs, var=var)

    # Stable categorical definition
    adata2.obs["condition"] = pd.Categorical(
        adata2.obs["condition"],
        categories=["MTUS1+", "MTUS1-"],
        ordered=True,
    )

    # Cast common covariates when present
    for c in ["batch_global", "batch_number_collapsed", "batch_number", "sra.library_selection"]:
        if c in adata2.obs.columns:
            adata2.obs[c] = adata2.obs[c].astype("category")

    # Cast continuous covariates when present
    if "prolif_score" in adata2.obs.columns:
        adata2.obs["prolif_score"] = pd.to_numeric(
            adata2.obs["prolif_score"], errors="coerce"
        )

    return adata2



# =========================
# 3) Harmonize factor names for PyDESeq2
# =========================
def harmonize_pydeseq2_names(adata, cols):
    """
    PyDESeq2 may internally replace '_' with '-'
    in factor names. Here we proactively rename obs columns
    and return the mapping.
    """
    mapping = {}
    for c in cols:
        if c in adata.obs.columns and "_" in c:
            mapping[c] = c.replace("_", "-")
    if mapping:
        adata.obs = adata.obs.rename(columns=mapping)
    return adata, mapping


# =========================
# 4) Prune design factors (missing/constant) + cast to category
# =========================
def prune_design_factors(adata, design_factors):
    """
    Drop design factors that are missing from adata.obs
    or constant (single level). Cast retained factors to 'category'.
    """
    keep = []
    dropped = {}
    for f in design_factors:
        if f not in adata.obs.columns:
            dropped[f] = "missing_in_obs"
            continue

        s = adata.obs[f]
        nun = s.nunique(dropna=True)

        if nun < 2:
            dropped[f] = f"constant_or_single_level (nunique={nun})"
            continue

        # Recommended for PyDESeq2 categorical covariates
        adata.obs[f] = adata.obs[f].astype("category")
        keep.append(f)

    return keep, dropped


# =========================
# 5) Design selection
# =========================
def get_design_base(dataset_name: str):
    """
    Select the base design per cohort.
    """
    if "Pooled" in dataset_name:  # pooled (per current naming convention)
        return ["batch_global", "condition"]
    if "TCGA" in dataset_name:
        return ["batch_number_collapsed", "condition"]
    if dataset_name == "SRP157974":
        return ["sra.library_selection", "condition"]
    return ["condition"]


# =========================
# 6) Prolif score (z-score within cohort for pooled)
# =========================
PROLIF_GENES = ["MKI67", "PCNA", "TOP2A","CCNB1","CCNB2","CDK1","CDC20", "MCM2","MCM4","MCM6","MCM7"]

def add_prolif_score(
    adata,
    genes=PROLIF_GENES,
    target_sum=1e6,
    zscore_within=True,
    zscore_group=None,   # e.g. "batch_global" for pooled analyses
    min_genes=1,
):
    genes_used = [g for g in genes if g in adata.var_names]
    adata.uns["prolif_genes_used"] = genes_used

    if len(genes_used) < min_genes:
        adata.obs["prolif_score"] = np.nan
        return adata

    idx = adata.var_names.get_indexer(genes_used)
    X = adata.X

    if sparse.issparse(X):
        lib = np.asarray(X.sum(axis=1)).ravel()
        lib_safe = np.where(lib == 0, np.nan, lib)
        Xg = X[:, idx]

        Xg_norm = Xg.multiply(1.0 / lib_safe[:, None]) * target_sum
        Xg_norm = Xg_norm.tocoo()
        Xg_norm.data = np.log1p(Xg_norm.data)
        Xg_norm = Xg_norm.tocsr()

        score = np.asarray(Xg_norm.mean(axis=1)).ravel()
    else:
        lib = X.sum(axis=1)
        lib_safe = np.where(lib == 0, np.nan, lib)
        Xg = X[:, idx]
        Xg_norm = (Xg / lib_safe[:, None]) * target_sum
        score = np.log1p(Xg_norm).mean(axis=1)

    adata.obs["prolif_score"] = score

    if zscore_within:
        if zscore_group is not None and zscore_group in adata.obs.columns:
            g = adata.obs[zscore_group].astype(str)
            mu = adata.obs.groupby(g)["prolif_score"].transform("mean")
            sd = adata.obs.groupby(g)["prolif_score"].transform("std")
            sd = sd.replace(0, np.nan)
            adata.obs["prolif_score"] = (adata.obs["prolif_score"] - mu) / sd
        else:
            mu = np.nanmean(adata.obs["prolif_score"].values)
            sd = np.nanstd(adata.obs["prolif_score"].values)
            if np.isfinite(sd) and sd > 0:
                adata.obs["prolif_score"] = (adata.obs["prolif_score"] - mu) / sd
            else:
                adata.obs["prolif_score"] = adata.obs["prolif_score"] - mu

    # Safety check: no infinite values.
    adata.obs["prolif_score"] = adata.obs["prolif_score"].replace([np.inf, -np.inf], np.nan)
    return adata



# =========================
# 7) RUN
# =========================
inference = DefaultInference(n_cpus=8)

# Expected contrast example:
# condition_a_tester = ("condition", "MTUS1-", "MTUS1+")

for dataset_name, dataset in datasets.items():
    print(dataset_name + "*" * 50)

    # Build global batch covariate
    dataset.anndata = add_global_batch(dataset_name, dataset)

    # 2. Add proliferation score
    dataset.anndata = add_prolif_score(dataset.anndata)

    # Clean metadata and keep only the two conditions
    dataset.anndata = clean_obs_for_design(dataset.anndata)

    # Choose base design
    design_base = get_design_base(dataset_name)

    # Harmonize obs column names used in the design (underscore -> hyphen)
    dataset.anndata, mp = harmonize_pydeseq2_names(dataset.anndata, design_base)
    design_base2 = [mp.get(c, c) for c in design_base]

    # Drop missing/constant factors and log what was removed
    design_used, dropped = prune_design_factors(dataset.anndata, design_base2)
    print("→ design used:", design_used)
    if dropped:
        print("→ dropped:", dropped)

    # Run PyDESeq2
    dds = DeseqDataSet(
        adata=dataset.anndata,
        design_factors=design_used,
        refit_cooks=True,
        inference=inference,
    )
    dds.deseq2()
    dataset.dds_base = dds

    stats = DeseqStats(
        dds,
        contrast=condition_a_tester,
        inference=inference,
    )
    stats.summary()
    dataset.stat_res_base = stats

    # Sort results by adjusted p-value
    res = stats.results_df.sort_values(by="padj")
    res["GeneName"] = res.index
    dataset.results_df_base = res


## qq affichages

In [ ]:
# Display dataframes cleanly without print.
from IPython.display import display

for name_dataset, dataset in datasets.items():
    print(name_dataset)
    #table = dataset.results_df[dataset.results_df['GeneName'].isin([name_gene,'MTUS1','GSK3B','MYC','HSP90AB1','HSP90AA1','DDX10', 'HYOU1','PSMB5','PSMB6','KAT5','PRMT1','WEE1','BRD4','BAG2', 'FHL2', 'IFI35', 'KIF2A','SDCCAG8', 'SKP1', 'TUBA4A'])].copy()
    table = dataset.results_df_base[dataset.results_df_base['GeneName'].isin([name_gene,'MTUS1','MYC','MYCN','MYCL'])].copy().T

    display(table.T)

# 4. Enrichment analysis

After performing DEA, we can use the obtained gene level statistics to perform enrichment analysis. Any statistic can be used, but we recommend using the t-values instead of logFCs since t-values incorporate the significance of change in their value. We will transform the obtained t-values stored in stats to a wide matrix so that it can be used by decoupler:

stat = log2FoldChange/lfcSE (standard error value )

 variance (VAR) of each gene by multiplying standard error (SE) by square root of total number of genes (n) and then squaring it :VAR=(SE*sqrt(n))^2

In [ ]:
for dataset in datasets.values():
    dataset.mat = dataset.results_df_base[['stat']].T.rename(index={'stat': condition_a_tester[1]+'.vs.'+condition_a_tester[2]})

In [ ]:
from gseapy import Msigdb
msig = Msigdb()

# Build a dataframe from a gp.get_library(name='MSigDB_Hallmark_2020') dictionary.
#MSigDB = pd.DataFrame.from_dict(gp.get_library(name='MSigDB_Hallmark_2020'), orient="index") # Previous option; may not be up to date.
#MSigDB = pd.DataFrame.from_dict(gp.get_library(name='WikiPathways_2024_Human'), orient="index") # Alternative option to test.
MSigDB = pd.DataFrame.from_dict(msig.get_gmt(category='h.all', dbver='2024.1.Hs'),orient="index")  # 50 gene sets
# Reshape the table so that gene symbols are in one column and gene-set names in another column.
# List containing all values.
gene_symbols = []
gene_set_names = []
k,l = MSigDB.shape
for i in range(0,k):
    for j in range(0,l):
        gene_symbols.append(MSigDB.iloc[i,j])
        gene_set_names.append(MSigDB.index[i])

# dataframe
MSigDB = pd.DataFrame({'geneset':gene_set_names,'genesymbol':gene_symbols})
# on remove les Noone
MSigDB = MSigDB[MSigDB['genesymbol'].isnull() == False]

In [ ]:

#datasets_5 = {'VUMC':VUMCa, 'GSE192341':GSE192341a,'GSE123845': GSE123845a,  'GSE202203': GSE202203a, 'TCGA': TCGAa,'BC_all_pooled': BC_all_pooled}#,'TNBC_all_pooled': TNBC_all_pooled}

df_pathways = pd.DataFrame()
for dataset_name,dataset in datasets.items():
    top_genes = dataset.results_df_base[np.isfinite(dataset.results_df_base['stat'])]
    # Restrict to genes in gene_list.
    #top_genes = top_genes[top_genes['GeneName'].isin(list_genes)]

    # Run GSEA
    enr_gsea_pvals = dc.get_gsea_df(
        df=top_genes,
        stat='stat',
        net=MSigDB,
        source='geneset',
        target='genesymbol'
    )
    # Sort by FDR-adjusted p-value.
    enr_gsea_pvals = enr_gsea_pvals.sort_values('FDR p-value')
    dataset.gsea_Hallmark = enr_gsea_pvals
    df_path = dataset.gsea_Hallmark[(dataset.gsea_Hallmark["NOM p-value"] < 0.05) & (dataset.gsea_Hallmark["FDR p-value"] < 0.05)].sort_values(by='NES', ascending=False)
    df_path_2 = df_path[["Term",'NES']]
    #df_path_3 = df_path[["Term",'NES','Leading edge']]
    # Set Term as the index.
    df_path_2.index = df_path_2["Term"]
    # Remove the Term column.
    df_path_2 = df_path_2.drop(columns=["Term"])
    df_path_2 = df_path_2.T
    df_path_2.index = [dataset_name]
    df_pathways = pd.concat([df_pathways,df_path_2],axis=0)

# Draw a heatmap of TF activities.
# Remove columns with more than two NaN values.
df_pathways_2 = df_pathways.dropna(axis=1,thresh=df_pathways.shape[0]-2)
#df_pathways_2 = df_pathways_2[['HALLMARK_E2F_TARGETS','HALLMARK_MYC_TARGETS_V1','HALLMARK_MYC_TARGETS_V2','HALLMARK_OXIDATIVE_PHOSPHORYLATION','HALLMARK_UV_RESPONSE_UP','HALLMARK_UNFOLDED_PROTEIN_RESPONSE','HALLMARK_DNA_REPAIR','HALLMARK_G2M_CHECKPOINT']]#,'HALLMARK_ESTROGEN_RESPONSE_EARLY','HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION','HALLMARK_ESTROGEN_RESPONSE_LATE','HALLMARK_XENOBIOTIC_METABOLISM',   'HALLMARK_COAGULATION','HALLMARK_INTERFERON_ALPHA_RESPONSE']]
# Replace NaN values with 0.
df_pathways_2 = df_pathways_2.fillna(0)

plt.figure(figsize=((14,10)))  # Optional: Adjust the size of the heatmap
# Calculate the min and max values for color scaling
vmin = df_pathways_2.min().min()  # Overall minimum value in the DataFrame
vmax = df_pathways_2.max().max()  # Overall maximum value in the DataFrame

ax = sns.heatmap(df_pathways_2.T, annot=True, cmap='coolwarm',  center=0, vmin=vmin, vmax=vmax, fmt='.2f',annot_kws={"fontsize": 15})
plt.xticks(fontsize=14, rotation=90)  # Adjust the fontsize and rotation angle for x-axis
plt.yticks(fontsize=14, rotation=0)   # Adjust the fontsize and rotation angle for y-axis
plt.title('Hallmark pathways, ' + condition_a_tester[1] + ' vs. ' + condition_a_tester[2], fontsize=15)
colorbar = ax.collections[0].colorbar  # Access the color bar object
colorbar.ax.tick_params(labelsize=14)  # Set the fontsize for the color bar labels

plt.show()




In [ ]:

df_pathways = pd.DataFrame()
for dataset_name,dataset in datasets.items():
    top_genes = dataset.results_df_base[np.isfinite(dataset.results_df_base['stat'])]
    # Restrict to genes in gene_list.
    #top_genes = top_genes[top_genes['GeneName'].isin(list_genes)]

    # Run GSEA
    enr_gsea_pvals = dc.get_gsea_df(
        df=top_genes,
        stat='stat',
        net=MSigDB,
        source='geneset',
        target='genesymbol'
    )
    # Sort by FDR-adjusted p-value.
    enr_gsea_pvals = enr_gsea_pvals.sort_values('FDR p-value')
    dataset.gsea_Hallmark = enr_gsea_pvals
    df_path = dataset.gsea_Hallmark.sort_values(by='NES', ascending=False)
    df_path_2 = df_path[["Term",'NES']]
    #df_path_3 = df_path[["Term",'NES','Leading edge']]
    # Set Term as the index.
    df_path_2.index = df_path_2["Term"]
    # Remove the Term column.
    df_path_2 = df_path_2.drop(columns=["Term"])
    df_path_2 = df_path_2.T
    df_path_2.index = [dataset_name]
    df_pathways = pd.concat([df_pathways,df_path_2],axis=0)
df_pathways_2 = df_pathways[['HALLMARK_MYC_TARGETS_V1','HALLMARK_MYC_TARGETS_V2','HALLMARK_OXIDATIVE_PHOSPHORYLATION','HALLMARK_UV_RESPONSE_UP','HALLMARK_UNFOLDED_PROTEIN_RESPONSE','HALLMARK_DNA_REPAIR','HALLMARK_E2F_TARGETS']]#,'HALLMARK_ESTROGEN_RESPONSE_EARLY','HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION','HALLMARK_ESTROGEN_RESPONSE_LATE','HALLMARK_XENOBIOTIC_METABOLISM',   'HALLMARK_COAGULATION','HALLMARK_INTERFERON_ALPHA_RESPONSE']]
#df_pathways_2 = df_pathways_2[['HALLMARK_MYC_TARGETS_V1','HALLMARK_MYC_TARGETS_V2','HALLMARK_OXIDATIVE_PHOSPHORYLATION','HALLMARK_UV_RESPONSE_DN','HALLMARK_UV_RESPONSE_UP','HALLMARK_UNFOLDED_PROTEIN_RESPONSE','HALLMARK_DNA_REPAIR','HALLMARK_G2M_CHECKPOINT','HALLMARK_ESTROGEN_RESPONSE_EARLY','HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION','HALLMARK_ESTROGEN_RESPONSE_LATE','HALLMARK_XENOBIOTIC_METABOLISM',   'HALLMARK_COAGULATION','HALLMARK_INTERFERON_ALPHA_RESPONSE']]
# Replace NaN values with 0.
df_pathways_2 = df_pathways_2.fillna(0)

plt.figure(figsize=((14,10)))  # Optional: Adjust the size of the heatmap
# Calculate the min and max values for color scaling
vmin = df_pathways_2.min().min()  # Overall minimum value in the DataFrame
vmax = df_pathways_2.max().max()  # Overall maximum value in the DataFrame

ax = sns.heatmap(df_pathways_2.T, annot=True, cmap='coolwarm',  center=0, vmin=vmin, vmax=vmax, fmt='.2f',annot_kws={"fontsize": 15})
plt.xticks(fontsize=14, rotation=90)  # Adjust the fontsize and rotation angle for x-axis
plt.yticks(fontsize=14, rotation=0)   # Adjust the fontsize and rotation angle for y-axis
plt.title('Hallmark pathways, ' + condition_a_tester[1] + ' vs. ' + condition_a_tester[2], fontsize=15)
colorbar = ax.collections[0].colorbar  # Access the color bar object
colorbar.ax.tick_params(labelsize=14)  # Set the fontsize for the color bar labels

In [ ]:
df_pathways

In [ ]:
from IPython.display import display
df_pathways = pd.DataFrame()
df_pv = pd.DataFrame()
df_FDR = pd.DataFrame()
for dataset_name,dataset in datasets.items():

    df_path = dataset.gsea_Hallmark.sort_values(by='NES', ascending=False)
    #dataset.gsea_Hallmark[ (dataset.gsea_Hallmark["FDR p-value"] < 0.05)].sort_values(by='NES', ascending=False)
    df_path_2 = df_path[["Term",'NES']]
    df_FDR_2 = df_path[["Term","FDR p-value"]]
    df_pval_2 = df_path[["Term","NOM p-value"]]

    # Set Term as the index.
    df_path_2.index = df_path_2["Term"]
    df_FDR_2.index = df_FDR_2["Term"]
    df_pval_2.index = df_pval_2["Term"]
    # Remove the Term column.
    df_path_2 = df_path_2.drop(columns=["Term"])
    df_path_2 = df_path_2.T
    df_FDR_2 = df_FDR_2.drop(columns=["Term"])
    df_FDR_2 = df_FDR_2.T
    df_pv_2 = df_pval_2.drop(columns=["Term"])
    df_pv_2 = df_pv_2.T

    # Change the index to use dataset names.
    if dataset_name == 'VUMC_GSE181466_SRP157974_GSE192341_SRP042620_GSE202203_TCGA':
        dataset_name = 'TNBC_All_pooled'
    elif dataset_name == 'VUMCa_GSE192341a_GSE123845a_GSE202203a_TCGAa' or dataset_name == 'VUMC_SRP042620_GSE181466_SRP157974_GSE192341_GSE202203_TCGA' or dataset_name == 'VUMCa_SRP042620a_GSE181466a_SRP157974a_GSE192341a_GSE202203a_TCGAa':
        dataset_name = 'BC_All_pooled'
    if dataset_name == 'SUM-52':
        dataset_name = 'Cell-lines'
    df_path_2.index = [dataset_name]
    df_FDR_2.index = [dataset_name]
    df_pv_2.index = [dataset_name]
    df_pathways = pd.concat([df_pathways,df_path_2],axis=0)
    df_FDR = pd.concat([df_FDR,df_FDR_2],axis=0)
    df_pv = pd.concat([df_pv,df_pv_2],axis=0)

# Create a dataframe with the minimum of df_FDR and df_pv.
df_pval = pd.DataFrame()
for i in range(0,df_FDR.shape[0]):
    for j in range(0,df_FDR.shape[1]):
        if df_FDR.iloc[i,j] > df_pv.iloc[i,j]:
            df_pval.loc[df_FDR.index[i],df_FDR.columns[j]] = df_FDR.iloc[i,j]
        else:
            df_pval.loc[df_FDR.index[i],df_FDR.columns[j]] = df_pv.iloc[i,j]

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

col = list(set(df_pathways_2.columns)| set(['HALLMARK_MYC_TARGETS_V1','HALLMARK_MYC_TARGETS_V2', 'HALLMARK_OXIDATIVE_PHOSPHORYLATION','HALLMARK_UV_RESPONSE_DN', 'HALLMARK_DNA_REPAIR','HALLMARK_UV_RESPONSE_UP' ]))
col = ['HALLMARK_MYC_TARGETS_V1','HALLMARK_MYC_TARGETS_V2','HALLMARK_DNA_REPAIR','HALLMARK_E2F_TARGETS', 'HALLMARK_OXIDATIVE_PHOSPHORYLATION','HALLMARK_UNFOLDED_PROTEIN_RESPONSE','HALLMARK_UV_RESPONSE_UP']# 'HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION','HALLMARK_ESTROGEN_RESPONSE_EARLY','HALLMARK_UV_RESPONSE_DN', 'HALLMARK_ESTROGEN_RESPONSE_LATE','HALLMARK_XENOBIOTIC_METABOLISM','HALLMARK_INTERFERON_GAMMA_RESPONSE','HALLMARK_KRAS_SIGNALING_UP','HALLMARK_INTERFERON_ALPHA_RESPONSE',	'HALLMARK_IL6_JAK_STAT3_SIGNALING' ]

col = ['HALLMARK_MYC_TARGETS_V1','HALLMARK_MYC_TARGETS_V2','HALLMARK_DNA_REPAIR',
       'HALLMARK_UV_RESPONSE_DN','HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION','HALLMARK_ESTROGEN_RESPONSE_EARLY',
        'HALLMARK_OXIDATIVE_PHOSPHORYLATION','HALLMARK_UNFOLDED_PROTEIN_RESPONSE',
       'HALLMARK_ESTROGEN_RESPONSE_LATE','HALLMARK_ANDROGEN_RESPONSE','HALLMARK_MYOGENESIS','HALLMARK_XENOBIOTIC_METABOLISM',
       #'HALLMARK_E2F_TARGETS','HALLMARK_UV_RESPONSE_UP',
       #'HALLMARK_BILE_ACID_METABOLISM','HALLMARK_COAGULATION',
       'HALLMARK_INTERFERON_GAMMA_RESPONSE','HALLMARK_INTERFERON_ALPHA_RESPONSE','HALLMARK_KRAS_SIGNALING_UP'
       #'HALLMARK_INFLAMMATORY_RESPONSE',#'HALLMARK_KRAS_SIGNALING_UP','HALLMARK_INTERFERON_ALPHA_RESPONSE',
       #  "HALLMARK_ALLOGRAFT_REJECTION",	'HALLMARK_IL6_JAK_STAT3_SIGNALING', "HALLMARK_MITOTIC_SPINDLE"
       ]# 'HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION','HALLMARK_ESTROGEN_RESPONSE_EARLY','HALLMARK_UV_RESPONSE_DN', 'HALLMARK_ESTROGEN_RESPONSE_LATE','HALLMARK_XENOBIOTIC_METABOLISM',,'HALLMARK_INTERFERON_ALPHA_RESPONSE',	'HALLMARK_IL6_JAK_STAT3_SIGNALING' ]


col = ['HALLMARK_MYC_TARGETS_V1','HALLMARK_MYC_TARGETS_V2','HALLMARK_DNA_REPAIR','HALLMARK_OXIDATIVE_PHOSPHORYLATION','HALLMARK_UNFOLDED_PROTEIN_RESPONSE',] #'HALLMARK_UV_RESPONSE_DN','HALLMARK_UV_RESPONSE_UP']
df_tf_renamed = df_pathways[col]
df_pv_plot = df_pval[col]


# Rename columns by removing "HALLMARK_" from the column names
df_tf_renamed = df_tf_renamed.rename(columns=lambda x: x.replace("HALLMARK_", ""))
#df_tf_renamed = df_tf_renamed.T.rename(columns={'TNBC_All5_pulled':'All_pulled'}).T
df_pv_renamed = df_pv_plot.rename(columns=lambda x: x.replace("HALLMARK_", ""))


# Calculate -log10(p-values)
log_pv = -np.log10(df_pv_renamed.T)
log_pv = log_pv.replace([np.inf, -np.inf], np.nan).fillna(0)

# Define sizes proportional to -log10(p-value)
min_size = 1   # Minimum circle size
max_size = 1000  # Maximum circle size
sizes = log_pv / log_pv.max().max() * (max_size - min_size) + min_size


# Function used to compute circle size from p-values.
def compute_circle_size(p_val):
    if p_val > 0.05:
        return 10  # Very small
    elif p_val > 0.01:
        return 50
    elif p_val > 0.001:
        return 100  # Small
    elif p_val > 0.0001:
        return 200  # Large
    else:
        return 500  # Very large

# Apply the function to each DataFrame element.
sizes =df_pv_renamed.T.applymap(compute_circle_size)

# Create the figure and main axis
fig, ax = plt.subplots(figsize=(9,7))


vmin =  -3.7 #max(df_tf_renamed.min().min(),-3)  # Overall minimum value in the DataFrame
vmax =  3.7#min(df_tf_renamed.max().max(),3)  # Overall maximum value in the DataFrame


# Draw circles.
x_labels =  df_tf_renamed.T.columns
y_labels =  df_tf_renamed.T.index
x_positions = range(len(x_labels))
y_positions = range(len(y_labels))



for y, row in enumerate(y_positions):
    for x, col in enumerate(x_positions):
        value =  df_tf_renamed.T.iloc[y, x]
        size = sizes.iloc[y, x]
        #print(f"value: {value}, vmin: {vmin}, vmax: {vmax}")
        #color = plt.cm.Reds((value - vmin) / (vmax - vmin))
        color = plt.cm.coolwarm((value - vmin) / (vmax - vmin)) 
        ax.scatter(x, -y, s=size, c=[color], edgecolor='k', alpha=0.8)

# Adjust axes and labels
ax.set_xticks(x_positions)
# Label size.
ax.tick_params(axis='x', labelsize=12)
ax.tick_params(axis='y', labelsize=12)
ax.set_xticklabels(x_labels, rotation=90)
ax.set_yticks([-i for i in y_positions])
ax.set_yticklabels(y_labels)
ax.invert_yaxis()
ax.set_aspect('equal')
ax.set_xlim(-0.5, len(x_labels) - 0.5)
ax.set_ylim(-len(y_labels) + 0.5, 0.5)

legend_sizes = [10, 50, 100, 200,500]  # Example sizes.
legend_labels = ["p > 0.05", "0.01 < p ≤ 0.05", "0.001 < p ≤ 0.01", "0.0001 < p ≤ 0.001", "p ≤ 0.0001"]
legend_circles = [plt.scatter([], [], s=size, color='gray', edgecolor='k', alpha=0.8) for size in legend_sizes]

# Create an axis for the legend.
#ax_legend1  = fig.add_axes([0.87, 0.2, 0.05, 0.2])  # Position (gauche, bas, largeur, hauteur)
#ax_legend1.axis("off")  # Hide unused axes.
#ax_legend1.legend(legend_circles, legend_labels, title="Adjusted p_value", loc="center", frameon=True)


# Add colorbar for NES below the plot
sm = plt.cm.ScalarMappable(cmap='coolwarm', norm=plt.Normalize(vmin=vmin, vmax=vmax))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, orientation='vertical', pad=0.05, fraction=0.02, aspect=15)
cbar.set_label("NES", fontsize=15)


# Add the global title at the top.
#fig.suptitle("Pathways NES Scores, summary of all datasets", fontsize=16, fontweight='bold')

# Refine layout to avoid overlaps and properly align legends
plt.subplots_adjust(bottom=.65, top=0.9)
plt.show()



In [ ]:
import numpy as np
import pandas as pd

def run_hallmark_gsea_from_deseq(df_res, MSigDB, stat_col="stat",
                                gene_col_out="genesymbol",
                                drop_na_padj=False,
                                sort_by="FDR p-value"):
    """
    Run pre-ranked GSEA (Hallmark / MSigDB) from a DESeq2 results table.

    Parameters
    ----------
    df_res : pd.DataFrame
        DESeq2 results (index = gene names) and a ranking column (default: 'stat').
    MSigDB : pd.DataFrame
        Decoupler network with columns: source='geneset', target='genesymbol' (or consistent with gene_col_out).
    stat_col : str
        Column used for ranking (e.g. Wald statistic).
    gene_col_out : str
        Name of the gene column expected by dc.get_gsea_df target (default: 'genesymbol').
    drop_na_padj : bool
        If True, drop rows with NA padj (optional).
    sort_by : str or None
        If provided, sort GSEA output by this column.

    Returns
    -------
    pd.DataFrame
        GSEA results.
    """
    df = df_res.copy()

    # Optionally drop NA adjusted p-values
    if drop_na_padj and "padj" in df.columns:
        df = df[df["padj"].notna()]

    # Keep finite ranking statistics
    df = df[np.isfinite(df[stat_col].to_numpy())].copy()

    # Provide gene symbols column for decoupler
    df[gene_col_out] = df.index.astype(str)

    enr = dc.get_gsea_df(
        df=df,
        stat=stat_col,
        net=MSigDB,
        source="geneset",
        target=gene_col_out,
    )

    if sort_by is not None and sort_by in enr.columns:
        enr = enr.sort_values(sort_by)

    return enr


In [ ]:
keep_terms = ['HALLMARK_MYC_TARGETS_V1','HALLMARK_MYC_TARGETS_V2','HALLMARK_DNA_REPAIR',
       'HALLMARK_UV_RESPONSE_DN','HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION','HALLMARK_ESTROGEN_RESPONSE_EARLY',
        'HALLMARK_OXIDATIVE_PHOSPHORYLATION','HALLMARK_UNFOLDED_PROTEIN_RESPONSE',
        'HALLMARK_ESTROGEN_RESPONSE_LATE','HALLMARK_ANDROGEN_RESPONSE','HALLMARK_MYOGENESIS','HALLMARK_XENOBIOTIC_METABOLISM',
       #'HALLMARK_E2F_TARGETS','HALLMARK_UV_RESPONSE_UP',
       #'HALLMARK_BILE_ACID_METABOLISM','HALLMARK_COAGULATION',
       'HALLMARK_INTERFERON_GAMMA_RESPONSE','HALLMARK_INTERFERON_ALPHA_RESPONSE','HALLMARK_KRAS_SIGNALING_UP'
       #'HALLMARK_INFLAMMATORY_RESPONSE',#'HALLMARK_KRAS_SIGNALING_UP','HALLMARK_INTERFERON_ALPHA_RESPONSE',
       #  "HALLMARK_ALLOGRAFT_REJECTION",	'HALLMARK_IL6_JAK_STAT3_SIGNALING', "HALLMARK_MITOTIC_SPINDLE"
       ]# 


keep_terms = [
    "HALLMARK_MYC_TARGETS_V1",
    "HALLMARK_MYC_TARGETS_V2",
    #"HALLMARK_E2F_TARGETS",
   # "HALLMARK_G2M_CHECKPOINT",
    "HALLMARK_DNA_REPAIR",
    "HALLMARK_OXIDATIVE_PHOSPHORYLATION",
    "HALLMARK_UNFOLDED_PROTEIN_RESPONSE",
    #"HALLMARK_UV_RESPONSE_UP",
]




df_nes_base = pd.DataFrame()
df_fdr_base = pd.DataFrame()


for dataset_name, dataset in datasets.items():

    # -------- base --------
    if hasattr(dataset, "results_df_base") and dataset.results_df_base is not None:
        gsea_base = run_hallmark_gsea_from_deseq(dataset.results_df_base, MSigDB, stat_col="stat")
        dataset.gsea_Hallmark_base = gsea_base

        gsea_base = gsea_base.set_index("Term")
        nes = gsea_base["NES"].reindex(keep_terms)
        fdr = gsea_base["FDR p-value"].reindex(keep_terms)

        df_nes_base = pd.concat([df_nes_base, nes.to_frame().T.assign(cohort=dataset_name)], axis=0)
        df_fdr_base = pd.concat([df_fdr_base, fdr.to_frame().T.assign(cohort=dataset_name)], axis=0)



# Cohort index.
for df in [df_nes_base, df_fdr_base]:
    if "cohort" in df.columns:
        df.set_index("cohort", inplace=True)

# Optional: fill missing values.
df_nes_base = df_nes_base.astype(float)
df_fdr_base = df_fdr_base.astype(float)

df_fdr_base

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
def make_star_annot(nes_df, fdr_df, alpha=0.05, fmt="{:.2f}"):
    annot = nes_df.copy()
    for r in nes_df.index:
        for c in nes_df.columns:
            v = nes_df.loc[r, c]
            q = fdr_df.loc[r, c]
            if pd.isna(v):
                annot.loc[r, c] = ""
            else:
                star = "*" if (pd.notna(q) and q < alpha) else ""
                annot.loc[r, c] = fmt.format(v) + star
    return annot


annot_base = make_star_annot(df_nes_base, df_fdr_base, alpha=0.05)

plt.figure(figsize=(9,3))
ax = sns.heatmap(
    df_nes_base.T,
    cmap="coolwarm",
    center=0,
    vmin=-3.5, vmax=3.5,          # Use the same scale for base/adjusted values when comparing them.
    annot=annot_base.T,
    fmt="",                       # Important: annot is already a string.
    annot_kws={"fontsize": 9},
    linewidths=0.2
)
plt.title("Hallmark NES, * = FDR < 0.05")
plt.tight_layout()
plt.show()


# 5. Proliferation score

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.stats import pearsonr, spearmanr

def _get_gene_logcpm_from_X(adata, gene="MTUS1", target_sum=1e6):
    """Return a log1p(CPM) vector for `gene` from adata.X raw counts."""
    if gene not in adata.var_names:
        raise ValueError(f"{gene} not in adata.var_names.")

    gidx = int(np.where(adata.var_names == gene)[0][0])
    X = adata.X

    if sp.issparse(X):
        lib = np.asarray(X.sum(axis=1)).ravel()
        gene_counts = np.asarray(X[:, gidx].todense()).ravel()
    else:
        lib = np.asarray(X.sum(axis=1)).ravel()
        gene_counts = np.asarray(X[:, gidx]).ravel()

    lib = np.where(lib == 0, np.nan, lib)
    cpm = (gene_counts / lib) * target_sum
    return np.log1p(cpm)

def corr_mtus1_prolif_by_cohort(
    adata,
    name_dataset,
    mtus1_gene="MTUS1",
    prolif_col="prolif_score",
    method=("pearson", "spearman")):

    # X: MTUS1 (logCPM)
    mtus1 = _get_gene_logcpm_from_X(adata, gene=mtus1_gene)
    # obs: prolif score
    prolif = pd.to_numeric(adata.obs[prolif_col], errors="coerce").to_numpy(dtype=float)

    # mask NA
    m = np.isfinite(mtus1) & np.isfinite(prolif)
    n = int(m.sum())

    out = {"cohort": name_dataset, "n": n, "pearson_r": np.nan, "pearson_p": np.nan,
                    "spearman_r": np.nan, "spearman_p": np.nan}
    if n > 3:
        if "pearson" in method:
            r, p = pearsonr(mtus1[m], prolif[m])
            out["pearson_r"] = float(r)
            out["pearson_p"] = float(p)

        if "spearman" in method:
            r, p = spearmanr(mtus1[m], prolif[m])
            out["spearman_r"] = float(r)
            out["spearman_p"] = float(p)

        return mtus1, prolif, n, out

df_corrs = []
for name_dataset, dataset in datasets.items():
    adata = dataset.anndata
    mtus1, prolif, n, out = corr_mtus1_prolif_by_cohort(
        adata,
        name_dataset=name_dataset,      
        mtus1_gene="MTUS1",
        prolif_col="prolif_score"
    )
    df_corrs.append(out)
df_corrs = pd.DataFrame(df_corrs)
df_corrs


In [ ]:
# plot
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(6, 6))
sns.scatterplot(x=mtus1, y=prolif)
# Regression line.
m, b = np.polyfit(mtus1[np.isfinite(mtus1) & np.isfinite(prolif)],
                  prolif[np.isfinite(mtus1) & np.isfinite(prolif)], 1)
plt.plot(mtus1, m * mtus1 + b, color='red')
plt.xlabel("MTUS1 expression (logCPM)")
plt.ylabel("Proliferation score (z-score)")
plt.title(f"Correlation MTUS1 vs. Proliferation in TNBC_All_pooled dataset\n"
          f"Pearson r={df_corrs.loc[7, 'pearson_r']:.2f} (p={df_corrs.loc[7, 'pearson_p']:.2e}), "
          f"Spearman r={df_corrs.loc[7, 'spearman_r']:.2f} (p={df_corrs.loc[7, 'spearman_p']:.2e})")
plt.axhline(0, color='gray', linestyle='--')
plt.grid()
plt.show()
# =========================